# Final Reproducible Pipeline — Cookie Cats

This notebook demonstrates **end-to-end reproducibility** of the Cookie Cats
A/B Testing & Player Retention Analysis project, following the **CRISP-DM**
methodology.

Every heavy-lifting function lives in the `src/` package; this notebook
orchestrates them in sequence so the full pipeline can be re-run with
**Kernel → Restart & Run All**.

## Step 0 — Environment Setup

Add the `src/` directory to `sys.path` so we can import our reusable modules.

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')

# Ensure src/ is importable from the notebook
src_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"Source path: {src_path}")
print("Setup complete ✓")

## Step 1 — Business Understanding & Data Loading

**Research Question:** Does moving the first gate in Cookie Cats from
level 30 to level 40 significantly affect 7-day player retention?

**Business Value:** Understanding gate placement helps game designers
optimise player engagement and monetisation strategy.

We load the raw dataset from `data/raw/cookie_cats.csv` using our
`processing` module.

In [ ]:
from processing import load_data, explore_data

df = load_data()
info = explore_data(df)

print(f"\nDataset Shape: {info['shape']}")
print(f"Columns: {info['columns']}")
print(f"Duplicates: {info['duplicates']}")
print(f"\nVersion Distribution:")
for k, v in info['version_distribution'].items():
    print(f"  {k}: {v:,}")

## Step 2 — Data Quality Audit

Before cleaning, we perform a **rigorous data audit** to understand:
- Schema validity (expected columns and types)
- Missing values and their distribution
- Outlier detection using both **IQR** and **Z-score** methods
- Value range constraints

This audit informs our cleaning decisions in the next step.

In [ ]:
from processing import data_audit

audit_results = data_audit(df)

# Display outlier summary
print(f"\n📊 Audit Summary:")
print(f"  Schema valid: {audit_results['schema']['schema_valid']}")
print(f"  Missing cells: {audit_results['missing_values']['total_missing_cells']}")
print(f"  IQR outliers: {audit_results['outliers_iqr']['outlier_count']:,} "
      f"({audit_results['outliers_iqr']['outlier_pct']}%)")

## Step 3 — Data Cleaning

Based on the audit findings, we now clean the data:
1. **Drop duplicates** on `userid`
2. **Cast** boolean retention columns to integers
3. **Cap outliers** in `sum_gamerounds` at the 99th percentile to reduce
   the influence of extreme values (audit showed 11%+ IQR outliers)

In [ ]:
from processing import preprocess_data

df_clean = preprocess_data(df, cap_outliers=True, cap_percentile=0.99)
df_clean.describe()

## Step 4 — Web Scraping & Data Augmentation

We scrape public Wikipedia pages to gather **industry benchmarks** for
mobile game retention rates. This external data contextualises Cookie Cats'
performance against genre averages.

**Sources scraped:**
- Wikipedia: Mobile game
- Wikipedia: Free-to-play
- Wikipedia: Video game industry
- Wikipedia: List of most-played mobile games by player count

**Tools used:** `requests` + `BeautifulSoup` (static HTML parsing)

We also compare **sequential vs. parallel** scraping performance using
`concurrent.futures.ThreadPoolExecutor`.

In [ ]:
from scraping import run_full_scraping_pipeline, create_genre_benchmarks

# Run the full scraping pipeline with performance comparison
scraping_result = run_full_scraping_pipeline(primary_df=df_clean)

### Scraping Performance Comparison

The table below compares sequential and parallel scraping times:

In [ ]:
import pandas as pd

perf = scraping_result['performance']
perf_df = pd.DataFrame([
    {'Method': 'Sequential', 'Time (s)': perf['sequential_time_s'], 'URLs': perf['urls_count']},
    {'Method': 'Parallel (4 workers)', 'Time (s)': perf['parallel_time_s'], 'URLs': perf['urls_count']},
])
perf_df['Speedup'] = [1.0, perf['speedup']]
print(perf_df.to_string(index=False))
print(f"\n→ Parallel scraping is {perf['speedup']:.2f}× faster")

### Industry Benchmarks (Scraped Data)

These genre-level benchmarks were derived from the scraped Wikipedia
articles' citation sources (GameAnalytics, Newzoo reports):

In [ ]:
benchmarks = scraping_result['genre_benchmarks']
print(benchmarks.to_string(index=False))

### Augmented Dataset

The scraped benchmarks are merged with the Cookie Cats data by assigning
the **match-3 genre** benchmarks to every row (Cookie Cats is a match-3
game). New columns include industry average retention rates and a
`retention_vs_industry` comparison feature.

In [ ]:
df_augmented = scraping_result['augmented_df']
print(f"Original shape:  {df_clean.shape}")
print(f"Augmented shape: {df_augmented.shape}")
print(f"\nNew columns added:")
new_cols = [c for c in df_augmented.columns if c not in df_clean.columns]
for col in new_cols:
    print(f"  - {col}: {df_augmented[col].iloc[0]}")

## Step 5 — Feature Engineering

We create new features to improve model performance:
- `gamerounds_bin` — engagement tier (inactive → hardcore)
- `high_engagement` — binary flag for 75th percentile players
- `retention_1_x_rounds` — interaction term
- `rounds_per_day_proxy` — daily engagement estimate

In [ ]:
from processing import engineer_features

df_feat = engineer_features(df_augmented)
df_feat.head()

## Step 6 — Exploratory Data Analysis

Visualise distributions and correlations to justify feature selection.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Game rounds distribution
sns.histplot(df_feat['sum_gamerounds'], bins=50, kde=True, ax=axes[0, 0])
axes[0, 0].set_title('Distribution of Game Rounds (Capped)')
axes[0, 0].set_xlabel('Total Game Rounds')

# Retention by version
retention_by_version = df_feat.groupby('version')[['retention_1', 'retention_7']].mean()
retention_by_version.plot(kind='bar', ax=axes[0, 1])
axes[0, 1].set_title('Retention Rates by Version')
axes[0, 1].set_ylabel('Retention Rate')
axes[0, 1].tick_params(axis='x', rotation=0)

# Engagement tier distribution
df_feat['gamerounds_bin'].value_counts().sort_index().plot(kind='bar', ax=axes[1, 0], color='teal')
axes[1, 0].set_title('Player Engagement Tiers')
axes[1, 0].set_xlabel('Engagement Level')
axes[1, 0].tick_params(axis='x', rotation=45)

# Correlation heatmap
numeric_cols = df_feat.select_dtypes(include='number').columns
corr = df_feat[numeric_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f', ax=axes[1, 1])
axes[1, 1].set_title('Feature Correlation Heatmap')

plt.tight_layout()
plt.savefig('../reports/figures/eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✓ Figure saved to reports/figures/eda_overview.png")

## Step 7 — Train / Test Split

Prepare features and target, then split 80/20 with stratification.

In [ ]:
from processing import prepare_modeling_data

X_train, X_test, y_train, y_test = prepare_modeling_data(df_feat)

## Step 8 — Model Training (sklearn Pipeline)

Train multiple classifiers inside `imblearn.Pipeline` with:
- `ColumnTransformer` (StandardScaler + OneHotEncoder)
- SMOTE for class imbalance
- Four models: Logistic Regression, Random Forest, XGBoost, Gradient Boosting

In [ ]:
from modeling import train_models

trained_models = train_models(X_train, y_train)
print(f"\n✓ {len(trained_models)} models trained")

## Step 9 — Hyperparameter Tuning

Tune the best-performing model(s) using `GridSearchCV` with 5-fold
cross-validation, optimising for ROC-AUC.

In [ ]:
from modeling import tune_hyperparameters

tuned_xgb = tune_hyperparameters(X_train, y_train, model_name='XGBoost')
tuned_rf = tune_hyperparameters(X_train, y_train, model_name='Random Forest')

## Step 10 — Model Evaluation

Evaluate all models using: **Accuracy, Precision, Recall, F1-score, ROC-AUC**.

| Metric | Why it matters |
|--------|---------------|
| Accuracy | Overall correctness (can be misleading with imbalanced classes) |
| Precision | "Of predicted retained, how many actually were?" |
| Recall | "Of actually retained, how many did we catch?" |
| F1 | Harmonic mean of Precision & Recall — balanced single metric |
| ROC-AUC | Threshold-independent discrimination ability |

In [ ]:
from modeling import (
    evaluate_all_models, metrics_summary_df, get_best_model,
    plot_model_comparison, plot_roc_curves, plot_confusion_matrices,
    evaluate_model,
)

# Evaluate all base models
eval_results = evaluate_all_models(trained_models, X_test, y_test)

# Add tuned XGBoost
eval_results['XGBoost (Tuned)'] = evaluate_model(tuned_xgb['best_pipeline'], X_test, y_test)

# Summary table
summary = metrics_summary_df(eval_results)
print("\n" + summary.to_string())

# Best model
best_name, best_score = get_best_model(eval_results)

In [ ]:
# Visualisations
plot_model_comparison(eval_results, save_path='../reports/figures/model_comparison.png')
plot_roc_curves(eval_results, y_test, save_path='../reports/figures/roc_curves.png')
plot_confusion_matrices(eval_results, save_path='../reports/figures/confusion_matrices.png')

## Step 11 — A/B Test Verification

Bootstrap analysis to statistically test whether the gate placement
change has a significant effect on 7-day retention.

In [ ]:
from processing import create_ab_groups, calculate_retention_metrics
from ab_testing import analyze_ab_test, plot_bootstrap_results, plot_retention_comparison

gate_30, gate_40 = create_ab_groups(df_feat)
metrics = calculate_retention_metrics(gate_30, gate_40)

print("Observed Retention Metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

# Bootstrap analysis
ab_results = analyze_ab_test(gate_30, gate_40, n_bootstraps=1000)
print(f"\np-value: {ab_results['p_value']:.4f}")
print(f"95% CI for difference: [{ab_results['confidence_intervals']['difference'][0]:.4f}, "
      f"{ab_results['confidence_intervals']['difference'][1]:.4f}]")

plot_bootstrap_results(ab_results, save_path='../reports/figures/bootstrap_results.png')
plot_retention_comparison(ab_results, save_path='../reports/figures/retention_comparison.png')

---
## ✅ Pipeline Complete

### Key Findings

1. **Gate placement matters:** Moving the gate from level 30 → 40 **decreases**
   7-day retention by ~0.8 percentage points (statistically significant).

2. **Industry context:** Cookie Cats' 7-day retention (~19%) is close to the
   match-3 genre industry benchmark (~22%), suggesting competitive performance.

3. **Best ML model:** The tuned classifier achieves ROC-AUC above the random
   baseline, confirming that player features have predictive power for retention.

### Reproducibility

This notebook can be re-run from scratch:
```
Kernel → Restart & Run All
```

All functions are imported from `src/` — no data processing logic lives in
this notebook.